In [ ]:

!pip install transformers datasets sentencepiece accelerate -q

In [ ]:
from transformers import RobertaTokenizer, RobertaForQuestionAnswering
from datasets import load_dataset
import torch

In [ ]:
dataset = load_dataset("squad") ## SQuAD data set
print(dataset)
print("\nExample:")
print("Context: ", dataset["train"][0]["context"][:200])
print("Question:", dataset["train"][0]["question"])
print("Answer:", dataset["train"][0]["answers"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

Example:
Context:  Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta
Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Answer: {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


In [ ]:
# load RoBERTa model and tokenizer
from transformers import AutoTokenizer, RobertaForQuestionAnswering

model_name = "deepset/roberta-base-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast = True) ## RobertaTokenizer -> AutoTokenizer with use_fast=True
model = RobertaForQuestionAnswering.from_pretrained(model_name)

print("Model loaded!")

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Model loaded!


In [ ]:
## select subset

small_train = dataset["train"].select(range(20000))
samll_val = dataset["validation"].select(range(2000))

print(f"Train size: {len(small_train)}")
print(f"Validation size: {len(samll_val)}")

Train size: 20000
Validation size: 2000


In [ ]:
# preprocess

MAX_LENGTH = 512

def preprocess(examples):
    questions = examples["question"]
    contexts = examples["context"]
    answers = examples["answers"]

    inputs = tokenizer(
        questions,
        contexts,
        max_length = MAX_LENGTH,
        truncation = True,
        padding = "max_length",
        return_offsets_mapping = True,
        return_tensors = None
    )

    start_positions = []
    end_positions = []

    for i, answer in enumerate(answers):
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        offset_mapping = inputs["offset_mapping"][i]

        start_pos = 0
        end_pos = 0

        for j, (start, end) in enumerate(offset_mapping):
            if start <= start_char < end:
                start_pos = j
            if start < end_char <= end:
                end_pos = j
                break
        
        start_positions.append(start_pos)
        end_positions.append(end_pos)
    
    inputs["start_positions"] = start_positions
    inputs["end_positions"]  = end_positions

    inputs.pop("offset_mapping")

    return inputs


In [ ]:
# applying preprocessing

tokenized_train = small_train.map(
    preprocess,
    batched = True,
    remove_columns = small_train.column_names

)
tokenized_val = samll_val.map(
    preprocess,
    batched = True,
    remove_columns = samll_val.column_names
)

print("tokenization done!")

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

tokenization done!


In [ ]:
# training arguments

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./roberta-doubt-solver",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none"
)

In [ ]:
# training setup

from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

class QATrainer(Trainer):  ## custom QATrainer banaya 
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model( ###compute_loss manually banaya — start_positions aur end_positions explicitly pass kar rahe hain
            end_positions=inputs["end_positions"],      
            input_ids = inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            start_positions=inputs["start_positions"], 
        )
        loss = outputs.loss
        return (loss,outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        with torch.no_grad():
            outputs = model(
                input_ids=inputs["input_ids"].to(model.device),
                attention_mask=inputs["attention_mask"].to(model.device),
                start_positions=inputs["start_positions"].to(model.device),
                end_positions=inputs["end_positions"].to(model.device)
            )
        loss = outputs.loss
        return (loss, None, None)

trainer = QATrainer(     ## model calculated loss properly now
    model = model,
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_val,
    processing_class = tokenizer,
    data_collator = data_collator,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


: 

In [ ]:
print(tokenized_train.column_names)
print(tokenized_train[0].keys())

['input_ids', 'attention_mask', 'start_positions', 'end_positions']
dict_keys(['input_ids', 'attention_mask', 'start_positions', 'end_positions'])


In [ ]:
# Save Model
model.save_pretrained("./roberta-doubt-solver")
tokenizer.save_pretrained("./roberta-doubt-solver")

print("Model 2 saved! ")

Model 2 saved! 


In [ ]:
#  Test Model
def answer_question(question, context):
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits) + 1
    
    answer = tokenizer.decode(inputs["input_ids"][0][start:end])
    return answer

# Test 
context = """
Deep Learning is a specialized area of Machine Learning that focuses on algorithms 
inspired by the structure and function of the human brain. Deep learning models 
automatically learn useful features from raw data through multiple layers of neural 
networks. Backpropagation is a method to calculate gradients in neural networks 
which helps in updating weights to minimize loss.
"""

question = "What is backpropagation?"

print("Answer:", answer_question(question, context))

Answer:  a method to calculate gradients in neural networks


In [ ]:
# Save Model
import shutil
from google.colab import files

# Pehle local save karo
model.save_pretrained("./t5-notes-summarizer")
tokenizer.save_pretrained("./t5-notes-summarizer")

# Phir zip banao aur download karo
shutil.make_archive("t5-notes-summarizer", 'zip', "./t5-notes-summarizer")
files.download("t5-notes-summarizer.zip")